In [6]:
import os
import sys
import time
import subprocess
import urllib.request
import psutil
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType, BooleanType

# ==============================================================================
# 0. KONFIGURASI ENVIRONMENT PYSPARK (WINDOWS)
# ==============================================================================
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.20.101-hotspot"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] = (
    os.path.join(os.environ["JAVA_HOME"], "bin") + ";" +
    os.path.join(os.environ["HADOOP_HOME"], "bin") + ";" +
    os.environ["PATH"]
)

# ==============================================================================
# 1. PERSIAPAN LINGKUNGAN & PENGUNDUHAN DATASET OTOMATIS (REPRODUCIBLE)
# ==============================================================================
os.makedirs("data", exist_ok=True)

crimes_csv_path = "data/chicago_crimes_2019_present.csv"
census_csv_path = "data/chicago_census_socioeconomic.csv"

# Endpoint bulk export resmi City of Chicago (berisi 100% data asli ~8.2+ juta baris, ~2.1 GB)
crimes_url = "https://data.cityofchicago.org/api/views/ijzp-q8t2/rows.csv?accessType=DOWNLOAD"
census_url = "https://data.cityofchicago.org/api/views/kn9c-c2s2/rows.csv?accessType=DOWNLOAD"
# Hapus file lama jika ukurannya masih di bawah 1 GB (agar file 50 MB sebelumnya tidak dianggap selesai)
# 1. Hapus file 97 MB yang terpotong sebelumnya
if os.path.exists(crimes_csv_path) and os.path.getsize(crimes_csv_path) < 1.5 * 1024 * 1024 * 1024:
    os.remove(crimes_csv_path)
if not os.path.exists(crimes_csv_path):
    print("Mengunduh 100% data (~2.1 GB) dengan transfer terkompresi... Mohon tunggu.")
    curl_cmd = (
        f'curl.exe --compressed -L --retry 5 -A "Mozilla/5.0" '
        f'-o "{crimes_csv_path}" "{crimes_url}"'
    )
    subprocess.run(curl_cmd, shell=True, check=True)
    print(f"\nSelesai! Ukuran file: {os.path.getsize(crimes_csv_path) / (1024*1024):.2f} MB")

if not os.path.exists(census_csv_path):
    print("Mengunduh Dataset Referensi (Chicago Socioeconomic Census)...")
    urllib.request.urlretrieve(census_url, census_csv_path)
    print("Selesai mengunduh tabel referensi!")

# ==============================================================================
# 2. INISIALISASI SPARK SESSION & DOKUMENTASI KONFIGURASI
# ==============================================================================
spark = (
    SparkSession.builder
    .appName("Chicago_Crime_BigData_Analytics")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .getOrCreate()
)

print("\n=== DOKUMENTASI LINGKUNGAN KOMPUTASI & SPARK ===")
print(f"Versi PySpark        : {spark.version}")
print(f"Master Mode          : {spark.sparkContext.master}")
print(f"Default Parallelism  : {spark.sparkContext.defaultParallelism}")
print(f"Total RAM Sistem     : {psutil.virtual_memory().total / (1024**3):.2f} GB")
print(f"Ukuran File Mentah   : {os.path.getsize(crimes_csv_path) / (1024**2):.2f} MB")

# ==============================================================================
# 3. DATA INGESTION, SCHEMA, RECORD COUNT, PARTISI, & CONTOH DATA AMAN
# ==============================================================================
df_crimes_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(crimes_csv_path)
)

# Standarisasi otomatis seluruh nama kolom (Case Number -> case_number, Primary Type -> primary_type, dll)
df_crimes_raw = df_crimes_raw.toDF(*[c.strip().lower().replace(" ", "_") for c in df_crimes_raw.columns])

df_census_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(census_csv_path)
)

print("\n=== PROFILING AWAL DATASET UTAMA (CRIMES) ===")
print(f"Jumlah Baris (Record Count) : {df_crimes_raw.count():,}")
print(f"Jumlah Kolom                : {len(df_crimes_raw.columns)}")
print(f"Jumlah Partisi RDD/DataFrame: {df_crimes_raw.rdd.getNumPartitions()}")

print("\n=== SCHEMA DATASET UTAMA ===")
df_crimes_raw.printSchema()

print("\n=== CONTOH 5 DATA AWAL (AMAN TANPA DATA PRIBADI) ===")
df_crimes_raw.select(
    "id", "case_number", "date", "block", "primary_type",
    "location_description", "arrest", "domestic", "community_area", "year"
).show(5, truncate=False)

# Standarisasi nama kolom tabel sensus agar siap di-join
df_census = (
    df_census_raw
    .withColumnRenamed("Community Area Number", "community_area_id")
    .withColumnRenamed("COMMUNITY AREA NAME", "community_area_name")
    .withColumnRenamed("PER CAPITA INCOME ", "per_capita_income")
    .withColumnRenamed("HARDSHIP INDEX", "hardship_index")
    .withColumnRenamed("PERCENT HOUSEHOLDS BELOW POVERTY", "pct_below_poverty")
    .select(
        F.col("community_area_id").cast(IntegerType()),
        F.col("community_area_name"),
        F.col("per_capita_income").cast(IntegerType()),
        F.col("hardship_index").cast(DoubleType()),
        F.col("pct_below_poverty").cast(DoubleType())
    )
    .filter(F.col("community_area_id").isNotNull())
)

# ==============================================================================
# 4. IMPLEMENTASI 6 OPERASI PYSPARK DATAFRAME API YANG BERMAKNA
# ==============================================================================

# --- OPERASI 1: SELECT & FILTER ---
# Memilih atribut kunci dan menyaring insiden yang memiliki informasi waktu & jenis kejahatan sah
df_filtered = (
    df_crimes_raw
    .select(
        F.col("id").cast(IntegerType()).alias("crime_id"),
        F.col("case_number"),
        F.col("date"),
        F.col("primary_type"),
        F.col("description"),
        F.col("location_description"),
        F.col("arrest").cast(BooleanType()).alias("is_arrest"),
        F.col("domestic").cast(BooleanType()).alias("is_domestic"),
        F.col("district").cast(IntegerType()).alias("district"),
        F.col("community_area").cast(IntegerType()).alias("community_area"),
        F.col("year").cast(IntegerType()).alias("year"),
        F.col("latitude").cast(DoubleType()).alias("latitude"),
        F.col("longitude").cast(DoubleType()).alias("longitude")
    )
    .filter(
        (F.col("primary_type").isNotNull()) &
        (F.col("date").isNotNull())
    )
)

# --- OPERASI 2: KOLOM TURUNAN (DERIVED COLUMNS) ---
# Mengekstrak Timestamp, Jam, Hari, dan mengkategorikan Shift Patroli Kepolisian
df_enriched = (
    df_filtered
    .withColumn("event_timestamp", F.to_timestamp(F.col("date"), "MM/dd/yyyy hh:mm:ss a"))
    .withColumn("hour_of_day", F.hour(F.col("event_timestamp")))
    .withColumn("day_of_week", F.date_format(F.col("event_timestamp"), "EEEE"))
    .withColumn(
        "patrol_shift",
        F.when((F.col("hour_of_day") >= 6) & (F.col("hour_of_day") < 14), "Shift 1 - Pagi/Siang (06:00-13:59)")
         .when((F.col("hour_of_day") >= 14) & (F.col("hour_of_day") < 22), "Shift 2 - Sore/Malam (14:00-21:59)")
         .otherwise("Shift 3 - Dini Hari (22:00-05:59)")
    )
)

print("\n=== HASIL OPERASI 1 & 2: FILTER DAN KOLOM TURUNAN SHIFT PATROLI ===")
df_enriched.select("crime_id", "event_timestamp", "primary_type", "hour_of_day", "day_of_week", "patrol_shift").show(5, truncate=False)

# --- OPERASI 3: GROUPBY & AGGREGATION + OPERASI 6: ORDERING ---
# Menghitung total insiden, jumlah penangkapan, dan Arrest Rate (%) per Kategori Kejahatan
df_crime_summary = (
    df_enriched
    .groupBy("primary_type")
    .agg(
        F.count("crime_id").alias("total_incidents"),
        F.sum(F.when(F.col("is_arrest") == True, 1).otherwise(0)).alias("total_arrests"),
        F.round(
            (F.sum(F.when(F.col("is_arrest") == True, 1).otherwise(0)) / F.count("crime_id")) * 100, 2
        ).alias("arrest_rate_pct")
    )
    .orderBy(F.col("total_incidents").desc())
)

print("\n=== HASIL OPERASI 3 & 6: AGREGASI & RANKING JENIS KEJAHATAN (TOP 10) ===")
df_crime_summary.show(10, truncate=False)

# --- OPERASI 4: JOIN ANTAR-TABEL ---
# Menggabungkan data kriminalitas dengan indikator sosial-ekonomi per Community Area
df_joined = (
    df_enriched
    .join(
        df_census,
        df_enriched["community_area"] == df_census["community_area_id"],
        how="inner"
    )
)

df_socio_crime = (
    df_joined
    .groupBy("community_area_id", "community_area_name", "hardship_index", "per_capita_income")
    .agg(
        F.count("crime_id").alias("total_crimes"),
        F.round(
            (F.sum(F.when(F.col("is_arrest") == True, 1).otherwise(0)) / F.count("crime_id")) * 100, 2
        ).alias("arrest_rate_pct")
    )
    .orderBy(F.col("total_crimes").desc())
)

print("\n=== HASIL OPERASI 4: JOIN KRIMINALITAS DENGAN DATA SOSIAL-EKONOMI WILAYAH ===")
df_socio_crime.show(10, truncate=False)

# --- OPERASI 5: WINDOW OPERATION ---
# Mencari Top 3 jenis kejahatan paling dominan di setiap Community Area beserta proporsinya
area_crime_counts = (
    df_joined
    .groupBy("community_area_name", "hardship_index", "primary_type")
    .agg(F.count("crime_id").alias("incident_count"))
)

window_area_rank = Window.partitionBy("community_area_name").orderBy(F.col("incident_count").desc())
window_area_total = Window.partitionBy("community_area_name")

df_top3_per_area = (
    area_crime_counts
    .withColumn("total_area_crimes", F.sum("incident_count").over(window_area_total))
    .withColumn("share_in_area_pct", F.round((F.col("incident_count") / F.col("total_area_crimes")) * 100, 2))
    .withColumn("crime_rank", F.row_number().over(window_area_rank))
    .filter(F.col("crime_rank") <= 3)
    .orderBy(F.col("total_area_crimes").desc(), F.col("crime_rank").asc())
)

print("\n=== HASIL OPERASI 5: WINDOW FUNCTION (TOP 3 KEJAHATAN PER COMMUNITY AREA TERPADAT) ===")
df_top3_per_area.show(15, truncate=False)

# ==============================================================================
# 5. MENJAWAB 2 PERTANYAAN ANALITIS DENGAN SPARK SQL
# ==============================================================================
df_joined.createOrReplaceTempView("vw_chicago_crimes_joined")

# --- SPARK SQL QUERY 1 (Menjawab Q1: Distribusi Shift Patroli & Arrest Rate) ---
sql_query_1 = """
SELECT
    patrol_shift,
    COUNT(crime_id) AS total_incidents,
    SUM(CASE WHEN is_arrest = true THEN 1 ELSE 0 END) AS total_arrests,
    ROUND(SUM(CASE WHEN is_arrest = true THEN 1 ELSE 0 END) * 100.0 / COUNT(crime_id), 2) AS arrest_rate_pct,
    SUM(CASE WHEN is_domestic = true THEN 1 ELSE 0 END) AS domestic_incidents
FROM vw_chicago_crimes_joined
GROUP BY patrol_shift
ORDER BY total_incidents DESC
"""

print("\n=== HASIL SPARK SQL 1: ANALISIS BEBAN INSIDEN & ARREST RATE PER SHIFT PATROLI ===")
spark.sql(sql_query_1).show(truncate=False)

# --- SPARK SQL QUERY 2 (Menjawab Q3: Komparasi Kriminalitas Berdasarkan Kelompok Hardship Index) ---
sql_query_2 = """
SELECT
    CASE
        WHEN hardship_index >= 60 THEN '1. High Hardship (Index >= 60)'
        WHEN hardship_index >= 30 AND hardship_index < 60 THEN '2. Medium Hardship (Index 30-59)'
        ELSE '3. Low Hardship (Index < 30)'
    END AS socioeconomic_tier,
    COUNT(DISTINCT community_area_id) AS num_communities,
    COUNT(crime_id) AS total_incidents,
    ROUND(COUNT(crime_id) * 1.0 / COUNT(DISTINCT community_area_id), 0) AS avg_incidents_per_community,
    ROUND(AVG(per_capita_income), 0) AS avg_per_capita_income_usd,
    ROUND(SUM(CASE WHEN is_arrest = true THEN 1 ELSE 0 END) * 100.0 / COUNT(crime_id), 2) AS arrest_rate_pct
FROM vw_chicago_crimes_joined
GROUP BY socioeconomic_tier
ORDER BY socioeconomic_tier ASC
"""

print("\n=== HASIL SPARK SQL 2: DAMPAK TINGKAT KESULITAN EKONOMI TERHADAP INSIDEN & PENANGKAPAN ===")
spark.sql(sql_query_2).show(truncate=False)

# ==============================================================================
# 6. PENGUKURAN WAKTU EKSEKUSI (BENCHMARKING) 2 PIPELINE SECARA KONSISTEN
# ==============================================================================
print("\n=== PENGUKURAN WAKTU EKSEKUSI (BENCHMARKING PIPELINE) ===")

# Bersihkan cache terlebih dahulu agar pengujian adil (Cold Run)
spark.catalog.clearCache()

# --- PIPELINE A: Membaca dan melakukan agregasi + Join langsung dari Raw CSV (Uncached) ---
start_pipeline_a = time.perf_counter()
raw_csv_a = spark.read.option("header", "true").option("inferSchema", "true").csv(crimes_csv_path)
raw_csv_a = raw_csv_a.toDF(*[c.strip().lower().replace(" ", "_") for c in raw_csv_a.columns])
res_a = (
    spark.read.option("header", "true").option("inferSchema", "true").csv(crimes_csv_path)
    .filter(F.col("Primary Type").isNotNull())
    .join(df_census, F.col("Community Area") == df_census["community_area_id"], "inner")
    .groupBy("community_area_name", F.col("Primary Type").alias("primary_type"))
    .count()
    .orderBy(F.col("count").desc())
    .collect()
)
duration_pipeline_a = time.perf_counter() - start_pipeline_a

# --- PIPELINE B: Menyimpan ke format Parquet (Silver Layer) + In-Memory Caching ---
parquet_path = "data/crimes_silver.parquet"
df_enriched.write.mode("overwrite").parquet(parquet_path)

df_parquet_cached = spark.read.parquet(parquet_path).cache()
df_parquet_cached.count()  # Materialisasi cache sebelum pengukuran query analitis

start_pipeline_b = time.perf_counter()
res_b = (
    df_parquet_cached
    .join(df_census, F.col("community_area") == df_census["community_area_id"], "inner")
    .groupBy("community_area_name", "primary_type")
    .count()
    .orderBy(F.col("count").desc())
    .collect()
)
duration_pipeline_b = time.perf_counter() - start_pipeline_b

print(f"Waktu Eksekusi Pipeline A (Raw CSV + Schema Inference + Disk I/O) : {duration_pipeline_a:.4f} detik")
print(f"Waktu Eksekusi Pipeline B (Parquet Columnar + In-Memory Cache)   : {duration_pipeline_b:.4f} detik")
print(f"Peningkatan Kecepatan (Speedup Factor)                            : {duration_pipeline_a / duration_pipeline_b:.2f}x lebih cepat")


=== DOKUMENTASI LINGKUNGAN KOMPUTASI & SPARK ===
Versi PySpark        : 4.3.0
Master Mode          : local[*]
Default Parallelism  : 12
Total RAM Sistem     : 15.71 GB
Ukuran File Mentah   : 1955.21 MB

=== PROFILING AWAL DATASET UTAMA (CRIMES) ===
Jumlah Baris (Record Count) : 8,671,708
Jumlah Kolom                : 22
Jumlah Partisi RDD/DataFrame: 16

=== SCHEMA DATASET UTAMA ===
root
 |-- id: integer (nullable = true)
 |-- case_number: string (nullable = true)
 |-- date: string (nullable = true)
 |-- block: string (nullable = true)
 |-- iucr: string (nullable = true)
 |-- primary_type: string (nullable = true)
 |-- description: string (nullable = true)
 |-- location_description: string (nullable = true)
 |-- arrest: boolean (nullable = true)
 |-- domestic: boolean (nullable = true)
 |-- beat: integer (nullable = true)
 |-- district: integer (nullable = true)
 |-- ward: integer (nullable = true)
 |-- community_area: integer (nullable = true)
 |-- fbi_code: string (nullable = true)
 